# Chapter 8 — Kimball Dimensional Modeling: Facts and Dimensions

**Source:** https://de101.startdataengineering.com/dw_table_types  
**Module:** Data Modeling

> **Prerequisite:** run `dw_tables.ipynb` first — it creates `dim_customer`, `fct_orders`, `fct_lineitem`, `wide_orders`, `wide_lineitem`, and `customer_outreach_metrics` in `prod_db`.

---

## Why data modeling matters

Moving data into an OLAP database is only half the job. The other half is **shaping** it so analytical queries are fast, consistent, and easy to write.

Source systems (your backend, APIs) normalise data for fast writes — foreign keys everywhere, no redundancy, minimal column count per table. That design is terrible for analytics: every business question requires 4+ joins.

**Kimball's Dimensional Model** is the dominant approach for redesigning that data for analytical use. It's been the industry standard since the 1990s and is what dbt models, Looker explores, and Airflow pipelines are almost always building toward.

---

## 8.1 Facts and Dimensions

Every Kimball warehouse is built from two types of tables:

| Table type | What a row represents | Examples |
|---|---|---|
| **Dimension** | A business *entity* — something that exists | customer, supplier, part, product, employee |
| **Fact** | A business *event* — something that happened | order placed, item shipped, payment received |

A useful mental model: **dimensions are nouns, facts are verbs.**

- `customer` is a noun → dimension table
- `order placed` is a verb → fact table
- `item shipped` is a verb → fact table

---

### Grain: the most important design decision

A table's **grain** (also called granularity or level) is the precise definition of what one row represents.

This is the most critical design decision in dimensional modeling. Everything else flows from it.

In the TPC-H dataset, two fact tables exist at different grains:

| Table | Grain | Key |
|---|---|---|
| `orders` | One row per **order** | `o_orderkey` |
| `lineitem` | One row per **item within an order** | `(l_orderkey, l_linenumber)` |

`lineitem` is at a finer grain than `orders`. You can always **roll up** from fine to coarse (sum lineitem prices to get order total), but you **cannot drill down** from coarse to fine (orders don't contain item-level discount information).

This is why both tables exist — they answer different questions at different levels of detail.

In [ ]:
%%sql
use prod_db

In [ ]:
%%sql
-- lineitem grain: one row per item in the order
-- Rolling up lineitem → orders: sum prices across all items in orderkey = 1
SELECT
    l_orderkey,
    ROUND(SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)), 2) AS computed_totalprice
FROM lineitem
WHERE l_orderkey = 1
GROUP BY l_orderkey

In [ ]:
%%sql
-- orders grain: one row per order — totalprice pre-computed at this grain
SELECT o_orderkey, o_totalprice
FROM orders
WHERE o_orderkey = 1

**What this shows:**

The `lineitem` fact table rolls up to produce the same `o_totalprice` stored in the `orders` fact table (small decimal difference due to `double` floating point precision — not a data error).

```
lineitem (fine grain — item level)
    ↓  SUM(price × discount × tax) GROUP BY orderkey
orders  (coarse grain — order level)
```

**Why keep both tables?**  
`orders` is sufficient for order-level questions ("how many orders did customer X place?"). But only `lineitem` can answer item-level questions ("which items were discounted?", "what is the average discount by part type?"). You need both grains.

**Rule:** never mix grains in one fact table. If you need both order-level and item-level measures, keep them in separate fact tables at their natural grains.

---

## 8.2 Dimension types: Full Snapshot and SCD2

Dimensions represent entities — but entities change over time. A customer moves to a different country. A supplier changes state. A product gets re-categorised.

How you handle these changes determines whether your historical analysis is accurate or silently wrong.

Kimball defines 7 dimension types. Two dominate in practice:

---

### Full Snapshot

The simplest approach: **overwrite (or reload) the entire dimension table each run**, optionally tagging each load with a `load_date`.

```
Run 1 (2024-01-01): load all 15,000 customers as of today
Run 2 (2024-02-01): load all 15,000 customers as of today (overwrites or inserts new snapshot)
```

**Pros:**
- Trivial to implement — `TRUNCATE` + `INSERT` or `CREATE TABLE AS SELECT`
- Easy time travel — filter by `load_date` to see what the dimension looked like at any past run
- Storage is cheap — keeping 6 months of snapshots is acceptable at modern prices

**Cons:**
- Wastes storage for slowly-changing data (99% of rows are identical between runs)
- Coarse time resolution — changes are captured at snapshot frequency (daily, weekly), not at the exact moment of change

**When to use:** when you need simplicity and the dimension doesn't change frequently, or when exact change timestamps aren't required for analysis.

---

### Slowly Changing Dimension Type 2 (SCD2)

Every change to an entity creates a **new row** in the dimension, rather than overwriting the old one. Each row tracks its validity window:

| Supplier_Key | Supplier_Code | Supplier_Name | Supplier_State | valid_from | valid_to | is_current |
|---|---|---|---|---|---|---|
| 123 | ABC | Acme Supply Co | **CA** | 2000-01-01 | 2004-12-22 | 0 |
| 123 | ABC | Acme Supply Co | **IL** | 2004-12-22 | NULL | 1 |

When the supplier moved from CA to IL on 2004-12-22:
- The old row gets `valid_to = 2004-12-22`, `is_current = 0` — it is not deleted
- A new row is inserted with `valid_from = 2004-12-22`, `valid_to = NULL`, `is_current = 1`

**This lets you answer time-sensitive questions accurately:**

```sql
-- "What state was this supplier in when order 9999 was placed in 2003?"
SELECT s.Supplier_State
FROM fct_orders o
JOIN dim_supplier s
  ON o.supplier_key = s.Supplier_Key
 AND o.order_date BETWEEN s.valid_from AND COALESCE(s.valid_to, CURRENT_DATE)
WHERE o.order_key = 9999
-- Returns 'CA' — correct for 2003, even though the supplier is now in IL
```

Without SCD2, you'd join on `supplier_key` alone and get `IL` — silently wrong.

**Three required columns in every SCD2 dimension:**

| Column | Purpose |
|---|---|
| `valid_from` | When this version of the entity became active |
| `valid_to` | When this version was superseded (NULL = currently active) |
| `is_current` | Boolean flag for fast "give me the current state" queries |

**Note on surrogate keys:** Kimball traditionally requires a surrogate key (auto-incrementing ID per row, not per entity) to be the join key in the fact table. In modern data systems, most teams skip this and join using the natural key + time range (`BETWEEN valid_from AND valid_to`), which is simpler and equally effective with columnar storage.

---

## 8.3 Building the dimension and fact tables

The TPC-H PARA model maps source tables to warehouse tables like this:

```
Source (OLTP / normalised)          Warehouse (OLAP / dimensional)
─────────────────────────           ──────────────────────────────
customer + nation + region    →     dim_customer
supplier + nation + region    →     dim_supplier  (not built in this course)
part                          →     dim_part      (not built in this course)
orders                        →     fct_orders
lineitem                      →     fct_lineitem
```

`dim_customer` denormalises three source tables (customer + nation + region) into one flat table. Every join that was required at query time in the OLTP system is now pre-computed at load time.

In [ ]:
%%sql
-- dim_customer: denormalised customer dimension
-- Pre-joins customer + nation + region — no join needed at query time
CREATE TABLE IF NOT EXISTS dim_customer AS
SELECT
    c.*,
    n.n_name    AS nation_name,
    n.n_comment AS nation_comment,
    r.r_name    AS region_name,
    r.r_comment AS region_comment
FROM customer c
LEFT JOIN nation n ON c.c_nationkey = n.n_nationkey
LEFT JOIN region r ON n.n_regionkey = r.r_regionkey

**Why LEFT JOIN, not INNER JOIN?**

Dimension tables should be complete — every customer row must appear, even if the nation or region data is missing (data quality issues happen). An `INNER JOIN` would silently drop customers without a valid nation, making the dimension incomplete and causing fact rows to lose their dimension context at query time.

**Rule for dimensions:** always `LEFT JOIN` when denormalising. Nulls in dimension attributes are acceptable; missing dimension rows are not.

In [ ]:
%%sql
-- fct_orders and fct_lineitem: fact tables
-- In this simple case they're identical to the source tables.
-- In production, fact tables often add derived columns (e.g. fiscal_quarter, days_to_ship)
-- and remove columns irrelevant to analytics.
CREATE TABLE IF NOT EXISTS fct_orders   AS SELECT * FROM orders;
CREATE TABLE IF NOT EXISTS fct_lineitem AS SELECT * FROM lineitem

In [ ]:
%%sql
-- Verify: row counts should match source tables exactly
SELECT 'fct_orders'   AS tbl, COUNT(*) AS rows FROM fct_orders
UNION ALL
SELECT 'fct_lineitem' AS tbl, COUNT(*) AS rows FROM fct_lineitem
UNION ALL
SELECT 'dim_customer' AS tbl, COUNT(*) AS rows FROM dim_customer

---

## 8.4 One Big Table (OBT)

As the number of fact and dimension tables grows, most analytical queries end up joining the same set of tables in the same way — every time. The OBT pattern eliminates that repeated join cost by pre-computing it.

**Definition:** a fact table LEFT-joined with all its relevant dimensions, at the grain of the fact table.

```sql
SELECT f.*, d.other_attributes
FROM fct_orders AS f
LEFT JOIN dim_customer AS d ON f.customer_key = d.customer_key
```

The result is a wide, denormalised table where every row has both the event (order) and all context (customer name, country, segment) already joined in.

**Key rule:** the OBT must have the **exact same grain as its base fact table**. `wide_orders` has one row per order (same as `fct_orders`). `wide_lineitem` has one row per line item (same as `fct_lineitem`). Never aggregate when building an OBT — that's what summary tables are for.

**Trade-offs:**

| | OBT | Star schema (fact + dims) |
|---|---|---|
| Query complexity | Simple — no joins needed | Complex — joins required each time |
| Storage | Higher — dimension data repeated per fact row | Lower — dimension data stored once |
| Update cost | High — full rebuild when dimension changes | Low — update only the dimension table |
| BI tool compatibility | Excellent — works with any tool without configuration | Requires semantic layer or view |

In modern data stacks, OBTs are the most practical choice for BI reporting layers — storage is cheap, and the query simplicity benefit is enormous.

In [ ]:
%%sql
-- wide_orders: OBT at order grain
-- One row per order, with all customer dimension attributes pre-joined
CREATE TABLE IF NOT EXISTS wide_orders AS
SELECT
    f.*,
    d.c_name         AS customer_name,
    d.c_address      AS customer_address,
    d.c_phone        AS customer_phone,
    d.c_acctbal      AS customer_acctbal,
    d.c_mktsegment   AS customer_segment,
    d.c_comment      AS customer_comment,
    d.nation_name,
    d.region_name
FROM fct_orders f
LEFT JOIN dim_customer d ON f.o_custkey = d.c_custkey

In [ ]:
%%sql
-- wide_lineitem: OBT at line item grain
-- For this course, lineitem has no dimension to join yet — kept as-is
CREATE TABLE IF NOT EXISTS wide_lineitem AS SELECT * FROM fct_lineitem

In [ ]:
%%sql
-- Demonstrate the OBT value: a rich analytical query with NO joins required
-- "Total revenue by customer segment and region, ordered by revenue"
SELECT
    customer_segment,
    region_name,
    COUNT(DISTINCT o_orderkey)           AS total_orders,
    ROUND(SUM(o_totalprice), 2)          AS total_revenue,
    ROUND(AVG(o_totalprice), 2)          AS avg_order_value
FROM wide_orders
GROUP BY customer_segment, region_name
ORDER BY total_revenue DESC
LIMIT 10

**Notice:** this query answers a complex business question — revenue by segment and region — with a single `GROUP BY` and no joins. In the normalised source schema, this would require joining `orders → customer → nation → region`.

This is the OBT payoff: the join cost is paid once at load time, and every subsequent query is simpler and faster.

---

## 8.5 Summary / pre-aggregated tables

OBTs are at the grain of the fact table — they still require aggregation at query time. For stakeholder-facing reporting, the final layer is the **summary table**: pre-aggregated at the specific grain the report needs.

**Two key benefits:**

1. **Consistent metric definitions** — the calculation lives in the data engineering codebase, not in every analyst's ad-hoc query. No more "why does my revenue number differ from yours?"
2. **No recomputation** — multiple stakeholders query the same pre-computed result. The expensive aggregation runs once per pipeline run, not on every dashboard load.

**The downside:** data is only as fresh as the last pipeline run. Summary tables are not real-time. This is an acceptable trade-off for most reporting use cases.

**In this course's pipeline:**
```
fct_lineitem (fine grain)  →  order_lineitem_metrics (per-order item count)
                                        ↓
wide_orders + order_lineitem_metrics  →  customer_outreach_metrics (per-customer aggregates)
```

In [ ]:
%%sql
-- Step 1: intermediate summary — items per order
-- Aggregates lineitem from item-grain to order-grain
CREATE TABLE IF NOT EXISTS order_lineitem_metrics AS
SELECT
    l_orderkey          AS order_key,
    COUNT(l_linenumber) AS num_lineitems
FROM wide_lineitem
GROUP BY l_orderkey

In [ ]:
%%sql
-- Step 2: customer_outreach_metrics — the final stakeholder-facing summary
-- One row per customer with pre-computed business KPIs
CREATE TABLE IF NOT EXISTS customer_outreach_metrics AS
SELECT
    wo.o_custkey                            AS customer_key,
    wo.customer_name,
    wo.customer_segment,
    wo.nation_name,
    wo.region_name,
    COUNT(DISTINCT wo.o_orderkey)           AS total_orders,
    ROUND(MIN(wo.o_totalprice), 2)          AS min_order_value,
    ROUND(MAX(wo.o_totalprice), 2)          AS max_order_value,
    ROUND(AVG(wo.o_totalprice), 2)          AS avg_order_value,
    ROUND(AVG(olm.num_lineitems), 2)        AS avg_items_per_order
FROM wide_orders wo
LEFT JOIN order_lineitem_metrics olm ON wo.o_orderkey = olm.order_key
GROUP BY wo.o_custkey, wo.customer_name, wo.customer_segment, wo.nation_name, wo.region_name

In [ ]:
%%sql
-- Inspect the result: top 10 customers by average order value
SELECT *
FROM customer_outreach_metrics
ORDER BY avg_order_value DESC
LIMIT 10

**The full pipeline in this chapter:**

```
Source tables (OLTP)
│
├── customer + nation + region  →  dim_customer       (dimension — entity)
├── orders                      →  fct_orders         (fact — event at order grain)
└── lineitem                    →  fct_lineitem        (fact — event at item grain)
                                         │
                                         ▼
                    fct_orders + dim_customer  →  wide_orders       (OBT — order grain)
                    fct_lineitem               →  wide_lineitem      (OBT — item grain)
                                         │
                                         ▼
                    wide_lineitem              →  order_lineitem_metrics   (summary)
                    wide_orders + metrics      →  customer_outreach_metrics (summary)
```

This is exactly the three-layer pattern (staging → core/OBT → marts/summary) that dbt implements in Chapter 11.

---

## 8.6 Exercises

### Exercise 1: What are the fact tables in the TPC-H data model?

**Answer:** `orders` and `lineitem`.

- `orders` — one row per **order placed** (`o_orderkey` is the unique event identifier). Grain: order.
- `lineitem` — one row per **item shipped as part of an order** (`l_orderkey` + `l_linenumber` is the composite key). Grain: line item.

Both represent business *events* (verbs), not entities. Each row is something that *happened*.

The other tables (`customer`, `supplier`, `part`, `partsupp`, `nation`, `region`) are all dimensional — they represent *things that exist*, not events.

---

### Exercise 2: What are the dimension tables in the TPC-H data model?

**Answer:** `customer`, `supplier`, `part`, `partsupp`, `nation`, `region`.

| Table | Entity it represents | Role in the model |
|---|---|---|
| `customer` | A buyer | Describes *who* placed the order |
| `supplier` | A seller/vendor | Describes *who* supplied the part |
| `part` | A product/item | Describes *what* was ordered |
| `partsupp` | A supplier-part combination | Bridge between supplier and part |
| `nation` | A country | Context for customer and supplier geography |
| `region` | A geographic region | Rolls up nations into regions |

Note that `nation` and `region` are **conformed dimensions** — they're shared context used by both `customer` and `supplier`. This is why `dim_customer` joins all three: `customer → nation → region`.

---

### Exercise 3: What source tables would you use to build a customer dimension?

**Answer:** `customer` + `nation` + `region`.

```sql
-- dim_customer denormalises all three into one flat table
SELECT
    c.*,
    n.n_name AS nation_name,
    r.r_name AS region_name
FROM customer c
LEFT JOIN nation n ON c.c_nationkey = n.n_nationkey
LEFT JOIN region r ON n.n_regionkey = r.r_regionkey
```

**Why not just `customer` alone?**  
`customer` stores `c_nationkey` (an integer FK), not the nation name. Any analyst query about "customers by country" would need to join `nation` every time. Pre-joining it into `dim_customer` eliminates that cost permanently.

**Why not include `orders` data in the customer dimension?**  
Orders are facts (events), not dimension attributes. Mixing them into the dimension would corrupt the grain of the dimension table and create update anomalies. Aggregated order metrics belong in a summary table (`customer_outreach_metrics`), not in the dimension.

---

## Key Takeaways

- **Dimensions = nouns (entities), Facts = verbs (events).** Every Kimball warehouse is built from these two types.
- **Grain is the most important design decision.** Define it precisely before writing any SQL. Never mix grains in one table.
- **Roll up, never drill down.** You can always aggregate from fine to coarse grain; you cannot recover detail from a coarse table.
- **Full Snapshot vs SCD2:** Full Snapshot is simple but coarse; SCD2 preserves the exact moment of change, enabling accurate historical joins.
- **SCD2 requires `valid_from`, `valid_to`, `is_current`.** Join on key + time range to get the attribute value at any point in history.
- **OBT = fact + all dimensions pre-joined.** Eliminates join complexity at query time. Must keep the same grain as the base fact table.
- **Always LEFT JOIN when denormalising dimensions.** Never drop fact rows because of missing dimension data.
- **Summary tables = pre-aggregated OBTs.** Consistent metric definitions + no recomputation at query time. Freshness is the trade-off.
- **The three-layer pipeline:** source → fact/dim → OBT → summary. This is what dbt models build in Chapter 11.

---

## Next: Chapter 9

Chapter 9 covers the **multi-hop architecture** (Bronze / Silver / Gold layers) — the production data flow pattern that organises how raw data moves through staging, transformation, and serving layers.

Source: https://de101.startdataengineering.com/dw_data_flow